# Lesson 7: Models and Choices

**Time:** ~40 minutes | **Cost:** $0 (no API calls)

## Why Different Models?

Different AI models exist for different tasks:

- **Claude Haiku** — fastest and cheapest, great for simple tasks
- **Claude Sonnet** — balanced: fast, affordable, handles tools and structured output well
- **Claude Opus** — most capable, best for complex reasoning, but slower and more expensive

No single model is "the best" — it depends on the task.

**Our project keeps it simple:** all agents use **Claude Sonnet**. One model, one API key, one provider. This makes the system easier to build, debug, and maintain.

## Speed / Cost / Quality Tradeoffs

Every model balances three factors. You can't have all three at maximum.

| Model | Speed | Cost | Best for |
|-------|-------|------|----------|
| Claude Haiku | Very fast | Very cheap | Simple tasks, classification, filtering |
| Claude Sonnet | Fast | Moderate | Tools, structured output, writing, analysis |
| Claude Opus | Slow | Expensive | Complex reasoning, nuanced judgment |

**Why Sonnet for everything in our project?**

Sonnet hits the sweet spot: it's fast enough for interactive use, affordable enough for batch processing, and capable enough for web search, article writing, image finding, and AIO analysis. Using one model everywhere avoids the complexity of managing multiple API providers.

In [ ]:
# Let's estimate the cost of our team per article

# Approximate costs per 1M tokens (as of 2025)
models = {
    "Claude Haiku": {"input": 0.80, "output": 4.00},
    "Claude Sonnet": {"input": 3.00, "output": 15.00},
    "Claude Opus": {"input": 15.00, "output": 75.00},
}

# Our team's token usage per article (approximate)
team_steps = [
    {"step": "Team Leader", "model": "Claude Sonnet", "input_tokens": 1500, "output_tokens": 500},
    {"step": "Content Writer", "model": "Claude Sonnet", "input_tokens": 3000, "output_tokens": 3500},
    {"step": "Image Finder", "model": "Claude Sonnet", "input_tokens": 4000, "output_tokens": 500},
    {"step": "AIO Analyzer", "model": "Claude Sonnet", "input_tokens": 3000, "output_tokens": 1500},
]

total_cost = 0
print("Cost breakdown per article:\n")
for step in team_steps:
    model = models[step["model"]]
    input_cost = step["input_tokens"] / 1_000_000 * model["input"]
    output_cost = step["output_tokens"] / 1_000_000 * model["output"]
    step_cost = input_cost + output_cost
    total_cost += step_cost
    print(f"  {step['step']:20s} ({step['model']:14s}): ${step_cost:.4f}")

print(f"\n  {'TOTAL':20s}                  : ${total_cost:.4f}")
print(f"\n  For 100 articles: ${total_cost * 100:.2f}")
print(f"  For 1000 articles: ${total_cost * 1000:.2f}")

## Our Architecture — One Model, Three Agents

All agents in our product use **Claude Sonnet**:

| Agent | Tools | Why Sonnet? |
|-------|-------|-------------|
| Content Writer | DataForSEO search + storage | Needs tools for web search. Good at long-form writing. |
| Image Finder | DataForSEO images + storage | Needs tools for image API. Reads and updates articles. |
| AIO Analyzer | AIO analysis functions | Needs tools for DataForSEO AIO API. Analyzes Google AI Overviews. |
| Team Leader | — (delegates to members) | Coordinates the team. Decides who handles each request. |

**Why not use different models for different agents?**

You could. For example, you might use Opus for complex analysis or Haiku for simple filtering. But for this product:
- **Sonnet handles everything well** — tools, writing, analysis
- **One API key** — only `ANTHROPIC_API_KEY` needed
- **Simpler debugging** — same model behavior everywhere
- **Cost is reasonable** — ~$0.10-0.30 per article

The tradeoff framework (speed/cost/quality) still matters when you build your own tools. Start simple with one model, then optimize later if needed.

In [ ]:
# What each Claude model can do

print("=== Claude Model Capabilities ===\n")

capabilities = {
    "Claude Haiku": {"tools": True, "output_schema": True, "both_together": True, "best_at": "Speed + low cost"},
    "Claude Sonnet": {"tools": True, "output_schema": True, "both_together": True, "best_at": "Balanced (our choice)"},
    "Claude Opus": {"tools": True, "output_schema": True, "both_together": True, "best_at": "Complex reasoning"},
}

for model, caps in capabilities.items():
    check = lambda x: "Yes" if x else "No"
    print(f"  {model:15s}  tools: {check(caps['tools']):3s}  schema: {check(caps['output_schema']):3s}  both: {check(caps['both_together']):3s}  | {caps['best_at']}")

print("\n=== Our Team Agent Requirements ===\n")

agents = [
    {"name": "Content Writer", "needs_tools": True, "needs_schema": False, "model": "Claude Sonnet"},
    {"name": "Image Finder", "needs_tools": True, "needs_schema": False, "model": "Claude Sonnet"},
    {"name": "AIO Analyzer", "needs_tools": True, "needs_schema": False, "model": "Claude Sonnet"},
    {"name": "Team Leader", "needs_tools": False, "needs_schema": False, "model": "Claude Sonnet"},
]

for agent in agents:
    needs = []
    if agent["needs_tools"]: needs.append("tools")
    if agent["needs_schema"]: needs.append("schema")
    needs_str = ", ".join(needs) if needs else "neither"
    print(f"  {agent['name']:18s} needs: {needs_str:10s} \u2192 {agent['model']}")

## Embeddings — A Different Kind of AI

Not all AI outputs are text. **Embeddings** turn text into lists of numbers that capture meaning.

GPS coordinates are a useful comparison: "Tokyo" and "Japan's capital" are close together on a map of meaning. "Baking bread" is far away.

```
"SEO"                        → [0.82, 0.15, 0.91, ...]
"search engine optimization" → [0.80, 0.17, 0.89, ...]  ← close!
"baking bread"               → [0.12, 0.88, 0.05, ...]  ← far away
```

Where embeddings are used:
- **Semantic search** — find articles similar to a query (not keyword matching)
- **Content clustering** — group similar articles together
- **RAG** (Retrieval-Augmented Generation) — find relevant documents to feed to an LLM

We don't use embeddings in this pipeline, but they're a key concept for future AI tools.

In [ ]:
# Simplified embedding demo (real embeddings use hundreds of dimensions)
# This shows the CONCEPT — real numbers come from embedding models

fake_embeddings = {
    "SEO optimization": [0.9, 0.8, 0.1, 0.2],
    "search engine ranking": [0.85, 0.75, 0.15, 0.25],
    "content marketing": [0.6, 0.5, 0.7, 0.3],
    "baking sourdough bread": [0.1, 0.1, 0.2, 0.9],
}

def similarity(a, b):
    """Simple dot product similarity (real systems use cosine similarity)"""
    return sum(x * y for x, y in zip(a, b))

print("Similarity scores (higher = more similar):\n")
base = "SEO optimization"
base_emb = fake_embeddings[base]

for text, emb in fake_embeddings.items():
    if text != base:
        score = similarity(base_emb, emb)
        bar = "\u2588" * int(score * 10)
        print(f"  \"{base}\" vs \"{text}\"")
        print(f"    Score: {score:.2f}  {bar}\n")

## Choosing the Right Model — Decision Guide

When you're deciding which model to use for an agent, follow this flowchart:

1. **Does the agent need tools** (web search, APIs)?
   - Yes → Must use a model that supports tools (all Claude models do)
2. **Does the agent need structured output** (output_schema)?
   - Yes → Must use a model that supports output_schema
3. **Is the task simple** (classification, filtering, yes/no)?
   - Yes → Consider **Haiku** (cheapest, fastest)
4. **Is the task complex reasoning** (planning, multi-step analysis)?
   - Yes → Consider **Opus** (most capable, but expensive)
5. **Default** → **Claude Sonnet** (best balance of speed, cost, capability)

Our project follows rule 5: Sonnet for everything. As you build your own tools, you might use Haiku for preprocessing or Opus for complex decisions.

In [ ]:
# Exercise: Match agents to models!
# For each scenario, which model would you choose and why?

scenarios = [
    {
        "name": "Email Writer Agent",
        "description": "Writes marketing emails from a brief",
        "needs_tools": False,
        "needs_schema": False,
        "priority": "Writing quality + low cost",
    },
    {
        "name": "Competitor Analyzer",
        "description": "Searches the web and returns structured competitor data",
        "needs_tools": True,
        "needs_schema": True,
        "priority": "Accuracy",
    },
    {
        "name": "Content Classifier",
        "description": "Categorizes articles into topics (returns a category label)",
        "needs_tools": False,
        "needs_schema": True,
        "priority": "Speed and cost",
    },
]

print("=== Model Selection Exercise ===\n")
for s in scenarios:
    print(f"Agent: {s['name']}")
    print(f"  Task: {s['description']}")
    print(f"  Needs tools: {'Yes' if s['needs_tools'] else 'No'}")
    print(f"  Needs schema: {'Yes' if s['needs_schema'] else 'No'}")
    print(f"  Priority: {s['priority']}")
    print(f"  Your choice: _______________")
    print()

print("Think about your answers, then check the suggested answers below!")

In [ ]:
# Suggested answers:

answers = {
    "Email Writer Agent": "Claude Sonnet \u2014 good writing quality at moderate cost. No tools or schema needed, so Haiku could also work if cost is critical.",
    "Competitor Analyzer": "Claude Sonnet \u2014 needs both tools AND schema. Sonnet handles both well at reasonable cost.",
    "Content Classifier": "Claude Haiku \u2014 simple task, schema needed but it's lightweight. Speed and cost matter most, and Haiku is the cheapest.",
}

print("=== Suggested Answers ===\n")
for agent, answer in answers.items():
    print(f"  {agent}: {answer}\n")

## Module 2 Summary — Understanding AI

| Lesson | Topic | Key takeaway |
|--------|-------|-------------|
| Lesson 5 | How LLMs Work | LLMs predict text token by token. They have knowledge cutoffs and can hallucinate. |
| Lesson 6 | Prompts and Context | Good prompts = Role + Task + Constraints + Examples. `instructions` is the system prompt. |
| Lesson 7 | Models and Choices | Different models for different tasks. Our project uses Sonnet everywhere for simplicity. |

You now understand **WHY** agents work the way they do. Module 3 is where you **BUILD** them.

**Next module:** Module 3 — Building Agents. You'll create your first AI agent, give it tools, make it return structured data, chain agents together, and build a mini pipeline.

## Exercise

No code needed — thought exercise:

Your SEO agency wants to build a new AI tool: a **"Content Audit Agent"** that:
- Crawls a website URL to read existing content (needs a tool)
- Analyzes the content for SEO issues
- Returns a structured report with scores and recommendations (needs schema)
- Must be affordable to run on hundreds of pages

**Questions:**
1. Which model would you choose? Why?
2. Could you use Haiku for this? Why or why not?
3. How would you keep costs down when processing hundreds of pages?

<details>
<summary>Click to reveal suggested answers</summary>

1. **Claude Sonnet** — it needs both tools (web crawling) and structured output (report schema), and Sonnet is affordable enough for bulk use.
2. **Maybe for part of it** — Haiku could do a quick pre-filter ("does this page even have SEO issues?"), but the detailed analysis and report generation benefits from Sonnet's deeper reasoning.
3. Use **Haiku** for a pre-filtering step (quick check if a page needs a full audit), then only send pages that need deep analysis to Sonnet. Also, keep prompts concise to minimize input tokens.

</details>

## Ready for Module 3? — Checkpoint

Module 3 is where it gets real: you'll create actual AI agents that call the Claude API.

- **Real API calls** — each cell costs a small amount (~$0.01-0.10)
- **You need an API key** — `ANTHROPIC_API_KEY` must be set in your `.env` file
- **Internet required** — agents call external APIs

Run the cell below to verify you're ready.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

# Check: Anthropic API key
print("Anthropic API key (for Claude)...")
key = os.getenv("ANTHROPIC_API_KEY", "")
if len(key) > 10:
    print(f"   PASS - Key found ({key[:8]}...)")
    print("\nYou're ready for Module 3!")
else:
    print("   FAIL - ANTHROPIC_API_KEY not set")
    print("   Fix: Add your key to the .env file in the project root")
    print("   Get one at: https://console.anthropic.com")
    print("\nFix the check above before starting Module 3.")